In [1]:
# Diffusion去噪可以有很多种类型网络，Unet, ResNet, Transformer，甚至最简单的MLP
# MLP 为例的DDPM

import math
import time
import numpy as np
import torch 
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class WeightedLoss(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, pred, targ, weighted=1.0):
        loss = self._loss(pred, targ)
        WeightedLoss = (loss * weighted).mean()

        return WeightedLoss
    
class WeightedL1(WeightedLoss):
    def _loss(self, pred, targ):
        return torch.abs(pred - targ)
    
class WeightedL2(WeightedLoss):
    def _loss(self, pred, targ):
        return F.mse_loss(pred, targ)
    
Losses = {
    "l1": WeightedL1,
    "l2": WeightedL2
}

In [6]:
def extract(a, t, x_shape):
    b, *_ = t.shape
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

In [3]:
class SinusoidaPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x: torch.Tensor):
        device = x.device
        half_dim = self.dim // 2 # 整除，作用：计算出一半的维度大小。如果总维度是 768。
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=1)
        return emb


In [ ]:
class MLP(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim, device, t_dim=16):
        super().__init__()

        self.action_dim = action_dim
        self.t_dim = t_dim
        self.device = device

        self.time_mlp = nn.Sequential(
            SinusoidaPosEmb(t_dim), # 自己写
            nn.Linear(t_dim, t_dim*2),
            nn.Mish(), # 在transformer中主要使用nn.Mish作为CV领域的激活函数， NLP用GELU
            nn.Linear(t_dim*2,t_dim)
        )

        input_dim = state_dim + action_dim + t_dim
        self.mid_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Mish(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Mish(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Mish()
        )

        self.final_layer = nn.Linear(hidden_dim, action_dim)

        self.init_weight()

    def init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, time, state):
        t_emb = self.time_mlp(time)
        x = torch.cat((x, state, t_emb), dim=1)
        x = self.mid_layer(x)
        x = self.final_layer(x)
        return x


In [ ]:
class Diffusion(nn.Module):
    def __init__(self, loss_type, beta_schedule = "linear", clip_denoised=True, predict_epsilon=True, **kwargs):
        super().__init__()
        self.state_dim = kwargs['obs_dim']
        self.action_dim = kwargs['act_dim']
        self.hidden_dim = kwargs['hidden_dim']
        self.T = kwargs["T"]

        self.clip_denoised = clip_denoised
        self.predict_epsilon = predict_epsilon

        self.device = torch.device(kwargs["device"])

        self.model = MLP(self.state_dim, self.action_dim, self.hidden_dim, self.device, self.T)

        if beta_schedule == "linear":
            betas = torch.linspace(0.0001, 0.02, self.T, dtype=torch.float32)

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, axis=0) # cumprod 累计乘积，这是DDPM里的at ba
        alphas_cumprod_prev = torch.cat((torch.ones(1), alphas_cumprod[: -1])) # at-1 ba

        # 使用静态buffer存储，方便后续快速调用
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cumprod", alphas_cumprod)
        self.register_buffer("alphas_cumprod_prev", alphas_cumprod_prev)

        # 前向过程
        self.register_buffer("sqrt_alphas_cumprod", torch.sqrt(alphas_cumprod))
        self.register_buffer("sqrt_one_minus_alphas_cumprod", torch.sqrt(1.0 - alphas_cumprod))

        # 反向过程：
        posterior_variance = (
            betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        )
        self.register_buffer("posterior_variance", posterior_variance)
        self.register_buffer("posterior_log_variance_clipped", torch.log(posterior_variance.clamp(min=1e-20)))
        # 还需要估计X0
        self.register_buffer("sqrt_recip_alphas_cumprod", torch.sqrt(1.0 / alphas_cumprod))
        self.register_buffer("sqrt_recipm_alphas_cumprod", torch.sqrt(1.0 / alphas_cumprod - 1))

        self.register_buffer("posterior_mean_coef1", betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod))
        self.register_buffer("posterior_mean_coef2", (1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod))

        self.loss_fn = Losses[loss_type]()

    def q_sample(self, x_start, t, noise):
        sample = (
            extract(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start + 
            extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise
        )
        return sample

    def q_posterior(self, x_start, x, t):
        posterior_mean = (
            extract(self.posterior_mean_coef1, t, x.shape) * x_start + extract(self.posterior_mean_coef2, t, x.shape) * x
        )
        posterior_variance  = extract(self.posterior_variance, t, x.shape)
        posterior_log_variance = extract(self.posterior_log_variance_clipped, t, x.shape)
        return posterior_mean, posterior_variance, posterior_log_variance

    def predict_start_from_noise(self, x, t, pred_noise):
        return (extract(self.sqrt_recip_alphas_cumprod, t, x.shape) * x - extract(self.sqrt_recipm_alphas_cumprod, t, x.shape) * pred_noise)

    def p_mean_variance(self, x, t, state):
        pred_noise = self.model(x, t, state)
        x_recon = self.predict_start_from_noise(x, t, pred_noise)
        x_recon.clamp_(-1, 1)
        model_mean, poster_variance, posterior_log_variance = self.q_posterior(x_recon, x, t)
        return model_mean, posterior_log_variance

    def p_sample(self, x: torch.Tensor, t, state):
        batch, *_, device = *x.shape, x.device
        model_mean, model_log_variance = self.p_mean_variance(x, t, state)
        noise = torch.randn_like(x)

        nonezero_mask = (1 - (t==0).float()).reshape(batch, *((1,) * (len(x.shape)-1)))
        return model_mean + nonezero_mask * (0.5 * model_log_variance).exp() * noise

    def p_sample_loop(self, state, shape, *args, **kwargs):
        device = self.device
        batch_size = state.shape[0]
        x = torch.randn(shape, device=device, requires_grad=False)

        for i in reversed(range(0, self.T)):
            t = torch.full((batch_size, ), i, device=device, dtype=torch.long)
            x = self.p_sample(x, t, state)

        return x

    def sample(self, state, *args, **kwargs):
        batch_size = state.shape[0]
        # DDPM开始会初始化一个噪声
        shape = [batch_size, self.action_dim]
        action = self.p_sample_loop(state, shape, *args, **kwargs)
        return action.clamp_(-1, 1) #防止超过边界

    def forward(self, state, *args, **kwargs):
        return self.sample(state, *args, **kwargs)
    
    def loss(self, x, state, weights=1.0):
        batch_size = len(x) #强化学习里维度就是动作大小
        t = torch.randint(0, self.T, (batch_size,), device=self.device).long()
        return self.p_losses(x, state, t, weights)
    
    def p_losses(self, x_start, state, t, weights):
        # x_start 就是x_0
        noise = torch.randn_like(x_start)
        x_noisy = self.q_sample(x_start, t, noise)
        x_recon = self.model(x_noisy, t, state)

        loss = self.loss_fn(x_recon, noise, weights)
        return loss


In [17]:
device = "cpu"
x = torch.randn(256, 2).to(device=device)
state = torch.randn(256, 11).to(device=device)

model = Diffusion(loss_type="l2", obs_dim=11, act_dim=2, hidden_dim=256, T=1000, device=device)
sample_action = model(state)

loss = model.loss(x, state)

print(f"action: {sample_action}, loss: {loss.item()}")

action: tensor([[nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan],
        [nan

In [ ]:
# test
batch = 1
seq_len = 10
dim = 10

x = torch.arange(seq_len, dtype=torch.float32)

pos_emb = SinusoidaPosEmb(dim=768)
emb = pos_emb(x)

emb.shape